# Bag-of-Reasoning (BoR) Training Flow

This notebook walks through the full BoR coder training pipeline on the **MATH-500** dataset:

1. **Sample reasoning traces** from two reasoning LLMs (`Qwen3-14B` and `Phi-4-reasoning`) using `vLLM`.
    - You only need to do this once if later you change the training parameters or want to train PoR instead
2. **Train the BoR coder** to learn a taxonomy that distinguishes the two models' reasoning traces.
3. **Evaluate** the learned taxonomy on a held-out test set.

## Step 1: Sample Reasoning Traces

Run the vLLM-based sampling script to generate reasoning traces for each model on MATH-500. Each command produces per-question output files under the designated output folder.

> **Note:** Adjust `--tensor_parallel_size` and `--gpu_memory_utilization` based on your GPU setup.

In [ ]:
!python sampling_py/MATH500/sample_MATH500_batched_vllm.py \
    --model_id "Qwen/Qwen3-14B" \
    --temperature 0.6 \
    --top_p 0.95 \
    --top_k 20 \
    --min_p 0 \
    --max_new_tokens 16384 \
    --batch_size 64 \
    --tensor_parallel_size 2 \
    --gpu_memory_utilization 0.9 \
    --download_dir "/n/holylabs/LABS/wattenberg_lab/Lab/pretrained_models/"

In [ ]:
!python sampling_py/MATH500/sample_MATH500_batched_vllm.py \
    --model_id "microsoft/Phi-4-reasoning" \
    --temperature 0.8 \
    --top_p 0.95 \
    --top_k 50 \
    --min_p 0 \
    --max_new_tokens 16384 \
    --batch_size 64 \
    --tensor_parallel_size 2 \
    --gpu_memory_utilization 0.9 \
    --download_dir "/n/holylabs/LABS/wattenberg_lab/Lab/pretrained_models/"

## Step 2: BoR Coder Training

The cells below are adapted from `coder_bor_full_training_procedure.py`. Instead of `argparse`, all configuration is set directly as variables.

In [ ]:
import os

import torch
from sklearn.model_selection import train_test_split

import pickle
import random
import numpy as np

from src.initial_codebook import INITIAL_CODEBOOK
from src import model_name_translator
from src.coder_bor_individual import CoderBORIndividual
from src.prompt_dataset import load_reasoning_traces, shuffle_outputs_and_labels

torch.manual_seed(10)
np.random.seed(0)
random.seed(10)

### Configuration

These are the hyperparameters for training the BoR coder.

In [ ]:
coder_model_id = "meta-llama/Llama-3.3-70B-Instruct" # HF ID of the LLM that you want to use as the annotator LLM
dataset_path = "/n/holylabs/LABS/wattenberg_lab/Users/yidachen/coders/outputs/MATH-500" # Path pointing to the folder that contains the reasoning traces 
compared_models = "Qwen3-14B,Phi-4-reasoning" # Name of the models to be compared by LOT, separate by comma, do not add spaces.

# You need at least 4 GPUs to use the LLaMa3.3-70B model as the annotator LLM
multi_gpus = 4
use_vllm = True

# Sampling hyperparameters for the coder / annotator models
temperature = 0.6
max_new_tokens = 32768
top_k = 50
top_p = 0.95
min_p = 0
frequency_penalty = 0.0

# Whether let the coder / annotator model to think before giving the annotations and updates
think_mode = "no"
think_budget = 0

# Number of samples used in 
num_warmup = 1 # Initializing the taxonomy
num_train = 49 # Expanding and validating the taxonomy
num_test = 200 # Test the classification performance of the taxonomy

# How many samples should the algorithm used to validate the new reasoning taxonomy (see section 3.2 of our paper)
accumulation_size = 40
# ALways use sampling_training if you want to run the algorithm described in our paper
sampling_training = True 

output_path = None                   # set to a custom path, or leave None for the default
job_id = "notebook_run"

batch_size = 8
evaluation_method = "logistic_regression"  # What kind of models will be trained on the PoR encoding (we used a linear logistic regression model in our paper) to predict the source LRM of the reasoning trace


# You could set an upper bound for how many reasoning behaviors to be included at most in the taxonomy
# You may want to set this limit if your annotation LLM has a small context window.
max_rule = 30


# Parameter for early exit of the training
# When LOT is overfitted or the training takes a long time to finish, you should try changing the numbers below
# Max number of training samples to use. After training the LOT on this number of samples, the algorithm will exit the training.
max_train_samples = 800
# Number of consecutive non-changed iteration before letting the training exit earlier
global_patience = 20
# Patience is deprecated. It should be set to 2 to align with the algorithm described in our paper.
patience = 2

# Whether to use imputation to extend the existing annotations to match with the notebook after update (see our paper section 3.3 for why we do this)
extend_with_nan = True
imputation_method = "knn"

# Normalize the BoR (frequency of reasoning) vectors based on the length of CoT so the 
normalize = False
normalization_method = "comparative"

# You could give LLM an initial codebook to iterate on (in our paper, we choose not to)
use_initial_codebook = "no"

# Use the parameter below to run the baselines in our paper
# Set run_type to "VML" for running the VML baseline
run_type = "Normal"
# Whether to run predefined / fixed taxonomy baseline
fixed_codebook_baseline = "no"


# Not used in our paper but can potentially stablize the taxonomy by removing the reasoning behaviors that LLM cannot consistently annotated on the same reasoning traces using different sampling seeds
reject_inconsistent_codes = False
averaged_eval = False
num_reruns = 5

seed = 123

### Set Up Output Paths & Instantiate the Coder

In [ ]:
if seed > 0:
    torch.manual_seed(seed)
    np.random.seed(seed)

model_options = compared_models.split(",")

dataset_folder = dataset_path.split('/')[-1].split('.')[0]
fixed_codebook_suffix = "-fixed-codebook-baseline" if fixed_codebook_baseline == "yes" else ""

if output_path is None:
    output_folder = f"coder_ckpt/{dataset_folder}-BoR{fixed_codebook_suffix}"
    intermediate_output_folder = os.path.join(output_folder, "intermediate_ckpt")
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(intermediate_output_folder, exist_ok=True)
else:
    os.makedirs(output_path, exist_ok=True)
    output_folder = output_path
    intermediate_output_folder = os.path.join(output_path, "intermediate_ckpt")
    os.makedirs(intermediate_output_folder, exist_ok=True)
    print("Use provided output folder:", output_path)

coder_type = "bor"

final_output_path = (
    f"{output_folder}/coder-{coder_type}-{normalization_method}-{coder_model_id.split('/')[-1]}"
    f"-compare-{'_'.join(model_options)}-on-{dataset_folder}-{job_id}.pkl"
)
intermediate_output_path = (
    f"{intermediate_output_folder}/coder-{coder_type}-{normalization_method}-{coder_model_id.split('/')[-1]}"
    f"-compare-{'_'.join(model_options)}-on-{dataset_folder}-{job_id}.pkl"
)

print("Output path:", final_output_path)

In [ ]:
base_folder = dataset_path

sampling_parameters = None  # vLLM uses SamplingParams internally

enabled_thinking = think_mode.lower() == "yes"
if enabled_thinking:
    print("Enable thinking mode")

CoderClass = CoderBORIndividual
print("Using CoderBORIndividual (Bag of Reasoning coder)")

max_input_tokens = 73728 if "llama" in coder_model_id.lower() else 24576
print("Max input tokens:", max_input_tokens)

initial_code_example = None

for m in model_options:
    if m not in model_name_translator:
        model_name_translator[m] = m

model_options = [model_name_translator[m] for m in model_options]
model_name_decoder = {v: k for k, v in model_name_translator.items()}

code_inst = None
correction_inst = None

if use_initial_codebook == "yes" or fixed_codebook_baseline == "yes":
    initial_codebook = INITIAL_CODEBOOK
else:
    initial_codebook = None

coder = CoderClass(
    coder_model_id,
    model_options,
    think_mode=enabled_thinking,
    cache_dir="/n/holylabs/LABS/wattenberg_lab/Lab/pretrained_models/",
    think_budget=think_budget,
    sampling_parameters=sampling_parameters,
    max_input_tokens=max_input_tokens,
    global_patience=global_patience,
    patience=patience,
    initial_code_example=initial_code_example,
    evaluation_method=evaluation_method,
    code_inst=code_inst,
    correction_inst=correction_inst,
    max_rule=max_rule,
    multi_gpus=multi_gpus,
    codebook=initial_codebook,
)

coder._extend_with_nan = extend_with_nan
coder._imputation_method = imputation_method
coder.max_train_samples = max_train_samples
coder.stop_criteria = ["max_rules", "global_patience", "max_train_samples"]

coder._normalize_vectors = normalize
coder._normalization_method = normalization_method

if fixed_codebook_baseline == "yes":
    print("Predefined Taxonomy Baseline: No Updates to the Codebook.")
    coder.no_update_to_codebook = True

coder.batch_update = False
coder.batch_update_size = 1

print(f"Coder initialized: {coder_type}")
print(f"Comparing models: {model_options}")

### Load Reasoning Traces & Prepare Dataset

In [ ]:
reasoning_traces = load_reasoning_traces(base_folder)

for model_name, traces in reasoning_traces.items():
    print("---------------")
    print(f"Model: {model_name}")
    print(f"Number of traces: {len(traces)}")
    for question_id, trace in traces.items():
        question = trace.get('question', '')
        thinking = trace.get('thinking', '')
        print(f"Question ID: {question_id}")
        print(f"Question: {question[:100]}")
        print(f"Reasoning Trace: {thinking[:100]}\n")
        break
    print("Empty Reasoning Traces:")
    for question_id, trace in traces.items():
        if len(trace.get('thinking', '')) < 1:
            print(question_id)
    print("\n")

In [ ]:
model_codename_0 = model_name_decoder[coder.model_options[0]]
model_codename_1 = model_name_decoder[coder.model_options[1]]
num_samples = len(reasoning_traces[model_codename_0])

all_idx = list(reasoning_traces[model_codename_0].keys())
all_idx = sorted(all_idx)

outputs = [
    [reasoning_traces[model_codename_0][idx]['thinking'],
     reasoning_traces[model_codename_1][idx]['thinking']]
    for idx in all_idx
]

questions = [
    reasoning_traces[model_codename_0][idx]['question']
    for idx in all_idx
]

labels = [
    [coder.model_options[0], coder.model_options[1]]
    for idx in all_idx
]

ids = [idx for idx in all_idx]

shuffled_outputs, shuffled_labels, shuffled_ids, shuffled_questions = shuffle_outputs_and_labels(
    outputs, labels, ids, questions
)

dataset = [
    {"outputs": o, "labels": l, "id": i, "question": q}
    for o, l, i, q in zip(shuffled_outputs, shuffled_labels, shuffled_ids, shuffled_questions)
]

warmup_dataset, remaining_dataset, _, remaining_labels = train_test_split(
    dataset, shuffled_labels, test_size=len(dataset) - num_warmup, random_state=42
)
train_dataset, eval_dataset, _, _ = train_test_split(
    remaining_dataset, remaining_labels,
    test_size=num_test, train_size=num_train, random_state=42, stratify=remaining_labels
)

print(f"Warmup dataset size: {len(warmup_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

for i in range(min(10, len(dataset))):
    print(dataset[i]["labels"])

### Warm Up the Coder

In [ ]:
warmup_attr_path = intermediate_output_path[:intermediate_output_path.rfind(".pkl")] + "_warmup_attr.pkl"

if not fixed_codebook_baseline == "yes":
    coder.warm_start(warmup_dataset, ckpt_path=warmup_attr_path)

### Training

In [ ]:
train_attr_path = intermediate_output_path[:intermediate_output_path.rfind(".pkl")] + "_trained_attr.pkl"

print(f"Train with accumulate observation training, accumulation size: {accumulation_size}")
coder.train(
    train_dataset,
    ckpt_path=train_attr_path,
    accumulate_observation_training=True,
    accumulation_size=accumulation_size,
    sampling_training=sampling_training,
    batch_size=accumulation_size,
)

### Evaluation

In [ ]:
eval_attr_path = intermediate_output_path[:intermediate_output_path.rfind(".pkl")] + "_evaled_attr.pkl"

if reject_inconsistent_codes:
    filtered_results, filtered_codebook = coder.reject_inconsistent_codes(
        random.sample(train_dataset, min(50, len(train_dataset))),
        num_reruns=num_reruns,
        modify_training_data=True,
    )

if averaged_eval:
    coder.eval_with_averaged_vectors(eval_dataset, num_reruns=num_reruns, variance_calculation=True)
else:
    print("Using batched evaluation for CoderBOR")
    coder.eval(eval_dataset, batched=True, batch_size=batch_size, ckpt_path=eval_attr_path)

coder.save_coder(final_output_path)
print(f"Coder saved to {final_output_path}")